In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="aaaaaa12121212/urdu-tts-speaker3-prepared", 
                  repo_type="dataset", local_dir="./urdu-tts-speaker3-prepared")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|██████████| 7/7 [00:03<00:00,  1.85it/s]


'/home/ubuntu/urdu-tts-speaker3-prepared'

In [3]:
files = glob('urdu-tts-speaker3-prepared/*/*.parquet')
files

['urdu-tts-speaker3-prepared/data/train-00003-of-00004.parquet',
 'urdu-tts-speaker3-prepared/data/train-00000-of-00004.parquet',
 'urdu-tts-speaker3-prepared/data/test-00000-of-00001.parquet',
 'urdu-tts-speaker3-prepared/data/train-00001-of-00004.parquet',
 'urdu-tts-speaker3-prepared/data/train-00002-of-00004.parquet']

In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [7]:
data = loop((files[:1], 0))

100%|██████████| 1421/1421 [01:23<00:00, 16.96it/s]


In [9]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1422/1422 [01:31<00:00, 15.55it/s]


In [10]:
len(data)

6319

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'urdu-tts-speaker3-prepared_audio/urdu-tts-speaker3-prepared-data-train-00003-of-00004_0.mp3',
 'text': 'سویرے ہی سویرے ہر گروپ کی میٹنگ ہوتی ہر روز کے کام کا پروگرام مرتب کیا جاتا اور سورج نکلنے سے پہلے ہی وہ ہیڈ کارٹر سے نکل جاتے وہ میلوں پیدل چلتے بستی کے لوگوں کے ساتھ فرش پر فسکڑا مار کر بیٹھ جاتے',
 'speaker': 'urdu-tts-speaker3-prepared_audio'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'urdu-tts-speaker3')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 181.24ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  62%|██████▏   |  553kB /  894kB, 65.8kB/s  
Processing Files (1 / 1): 100%|██████████|  894kB /  894kB,  104kB/s  
Processing Files (1 / 1): 100%|██████████|  894kB /  894kB, 97.2kB/s  
New Data Upload: 100%|██████████|  894kB /  894kB, 97.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.54s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/d07cc85abb99578839ba8bbbf7a0acb09682c1e0', commit_message='Upload dataset', commit_description='', oid='d07cc85abb99578839ba8bbbf7a0acb09682c1e0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('urdu-tts-speaker3-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq urdu-tts-speaker3-prepared_audio.zip urdu-tts-speaker3-prepared_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS urdu-tts-speaker3-prepared_audio.zip --repo-type=dataset

In [19]:
# !zip -rq urdu-tts-speaker3-prepared_audio_neucodec.zip urdu-tts-speaker3-prepared_audio_neucodec

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS urdu-tts-speaker3-prepared_audio_neucodec.zip --repo-type=dataset